# Lesson 5 — Train/Test Split and First Classifier

Read `README.md` (or `README.pl.md`) first — it has Meridian Outlet's framing for today. You built a baseline that never caught a return; now fit a real model and see if it does any better.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from task import load_and_merge_orders

df = load_and_merge_orders()
df.shape

## Split before you look

With only 14% of orders returned, a careless split could give the test set a very different return rate than the train set.

In [ ]:
from task import split_orders

train_df, test_df = split_orders(df)
len(train_df), len(test_df), train_df["is_returned"].mean(), test_df["is_returned"].mean()

## Wait — does this test set contain "new" customers?

Not necessarily. This is a row-level split: it doesn't know or care whether the same customer shows up on both sides. Check how much overlap there actually is.

In [ ]:
row_overlap = set(train_df["customer_id"]) & set(test_df["customer_id"])
pct_test_rows_from_known_customers = test_df["customer_id"].isin(row_overlap).mean()
len(row_overlap), pct_test_rows_from_known_customers

## Fit your first real classifier

Using the three signals you found in Lesson 3: discount percent, return history, and account age.

In [ ]:
from task import fit_classifier

model = fit_classifier(train_df)
model.coef_

## Predict at the default threshold and check the confusion matrix

Compare tp here to Lesson 4's baseline.

In [ ]:
from sklearn.metrics import confusion_matrix
from task import predict_return

predicted = predict_return(model, test_df)
confusion_matrix(test_df["is_returned"], predicted, labels=[0, 1])

## But look closer — the label alone doesn't tell the whole story

`predict_return` collapses every prediction to a hard 0 or 1 at the 0.5 threshold. Before you conclude the model learned nothing, look at the actual predicted probabilities.

In [ ]:
from task import FEATURE_COLUMNS

proba = model.predict_proba(test_df[FEATURE_COLUMNS])[:, 1]
proba.min(), proba.max(), proba.mean()

## Same model, both splits

The row-level split above left 90 customers on both sides. Compare it to a group-level split, where no customer appears in both train and test, and see how different the answer actually is.

In [ ]:
from task import split_orders_by_customer

group_train_df, group_test_df = split_orders_by_customer(df)
group_overlap = set(group_train_df["customer_id"]) & set(group_test_df["customer_id"])
len(group_train_df), len(group_test_df), len(group_overlap)

In [ ]:
from sklearn.metrics import roc_auc_score

group_model = fit_classifier(group_train_df)
group_pred = predict_return(group_model, group_test_df)
group_proba = group_model.predict_proba(group_test_df[FEATURE_COLUMNS])[:, 1]

row_cm = confusion_matrix(test_df["is_returned"], predicted, labels=[0, 1])
group_cm = confusion_matrix(group_test_df["is_returned"], group_pred, labels=[0, 1])
row_auc = roc_auc_score(test_df["is_returned"], proba)
group_auc = roc_auc_score(group_test_df["is_returned"], group_proba)

print("row split  : cm=", row_cm.tolist(), " auc=", row_auc)
print("group split: cm=", group_cm.tolist(), " auc=", group_auc)

## Your notes

Write down: given the probability range you just saw, is this model actually useless, or is 0.5 simply the wrong threshold for Meridian Outlet's problem? What do you think would happen to the confusion matrix if you lowered the threshold?

Also: why are the row-split and group-split results above close rather than dramatically different? What property of `previous_returns_count`/`account_age_days` (check `data/generate.py`) makes that less surprising than it might first seem?